# Notebook 04 — Detection des gaspillages

In [1]:
import pandas as pd
from pathlib import Path
df = pd.read_csv(Path('../../ressources/datasets/cur_sample.csv'), parse_dates=['usage_date'])
account_labels = {'111111111111':'prod','222222222222':'staging','333333333333':'dev','444444444444':'sandbox'}
df['account_name'] = df['account_id'].map(account_labels)
df['month'] = df['usage_date'].dt.to_period('M')
df['is_weekend'] = df['usage_date'].dt.dayofweek >= 5
print('Dataset charge :', len(df), 'lignes')

Dataset charge : 2523 lignes


## Heuristique 1 — EBS sans EC2 actif le meme jour

In [2]:
days_with_ec2 = df[df['service'] == 'AmazonEC2']['usage_date'].unique()
ebs_no_ec2 = df[
    (df['service'] == 'AmazonEBS') &
    (~df['usage_date'].isin(days_with_ec2))
]
print(f'Cout EBS sans EC2 actif : ${ebs_no_ec2.unblended_cost.sum():,.2f}')
if len(ebs_no_ec2) == 0:
    print('Aucun jour sans EC2 — heuristique non declenchee sur ce dataset')

Cout EBS sans EC2 actif : $0.00
Aucun jour sans EC2 — heuristique non declenchee sur ce dataset


## Heuristique 2 — Ressources constantes 24/7 en environnement dev

In [3]:
dev_daily = df[df['account_name'] == 'dev'].groupby(['service', 'usage_date'])['unblended_cost'].sum().reset_index()
dev_stats = dev_daily.groupby('service')['unblended_cost'].agg(['mean', 'std', 'count']).reset_index()
dev_stats['cv'] = (dev_stats['std'] / dev_stats['mean']).round(3)
constant = dev_stats[dev_stats['cv'] < 0.1].sort_values('mean', ascending=False)
print('Services avec cout constant en dev (CV < 10%) :')
print(constant[['service', 'mean', 'cv']].to_string(index=False))
print(f'\nCout mensuel estime gaspille : ${constant["mean"].sum() * 8:,.2f} (week-ends inclus)')

Services avec cout constant en dev (CV < 10%) :
Empty DataFrame
Columns: [service, mean, cv]
Index: []

Cout mensuel estime gaspille : $0.00 (week-ends inclus)


## Heuristique 3 — Cout week-end vs semaine en non-prod

In [4]:
non_prod = df[df['account_name'].isin(['dev', 'sandbox'])]
we = non_prod.groupby('is_weekend')['unblended_cost'].mean()
ratio = we[True] / we[False]
print(f'Cout moyen semaine  : ${we[False]:,.2f}')
print(f'Cout moyen week-end : ${we[True]:,.2f}')
print(f'Ratio week-end/semaine : {ratio:.2f}')
if ratio > 0.8:
    savings = (we[True] - we[False] * 0.1) * 8 * 4
    print(f'=> Gaspillage : les envs non-prod tournent le week-end')
    print(f'=> Economie potentielle si arret week-end : ${savings:,.2f}/mois')

KeyError: True

## Heuristique 4 — Top 10 usage_type avec cout > $1000 et tendance decroissante

In [ ]:
monthly = df.groupby(['usage_type', 'month'])['unblended_cost'].sum().reset_index()
monthly['month_num'] = monthly['month'].dt.month
top_usage = df.groupby('usage_type')['unblended_cost'].sum()
top_usage = top_usage[top_usage > 1000].index

declining = []
for ut in top_usage:
    data = monthly[monthly['usage_type'] == ut].sort_values('month_num')
    if len(data) >= 2 and data['unblended_cost'].iloc[-1] < data['unblended_cost'].iloc[0]:
        declining.append({'usage_type': ut,
                          'cout_debut': data['unblended_cost'].iloc[0],
                          'cout_fin': data['unblended_cost'].iloc[-1],
                          'baisse_%': round((1 - data['unblended_cost'].iloc[-1]/data['unblended_cost'].iloc[0])*100, 1)})

if declining:
    print('Usage types > $1000 avec tendance decroissante :')
    print(pd.DataFrame(declining).sort_values('baisse_%', ascending=False).to_string(index=False))
else:
    print('Aucun usage_type > $1000 avec tendance decroissante')

## Heuristique 5 — Instances EC2 potentiellement surdimensionnees

In [5]:
ec2 = df[df['service'] == 'AmazonEC2'].copy()
large = ec2[ec2['usage_type'].str.contains('xlarge|2xlarge|4xlarge', na=False)]
print('Instances EC2 de grande taille (potentiellement surdimensionnees) :')
print(large.groupby('usage_type')['unblended_cost'].sum().sort_values(ascending=False).to_string())
print(f'\nCout total grandes instances : ${large.unblended_cost.sum():,.2f}')

Instances EC2 de grande taille (potentiellement surdimensionnees) :
usage_type
BoxUsage:c5.2xlarge    9848.0585
BoxUsage:m5.xlarge     5781.9305

Cout total grandes instances : $15,629.99


## Exercice 4 — Heuristique 6 : croissance MoM > 50%

In [6]:
def detect_high_growth(df: pd.DataFrame, threshold_pct: float = 50.0) -> pd.DataFrame:
    """
    Detecte les services dont la croissance MoM (Month over Month) depasse threshold_pct%
    et qui ne sont pas tagges tag_project=growth.
    """
    monthly_svc = df.groupby(['service', 'month'])['unblended_cost'].sum().reset_index()
    monthly_svc = monthly_svc.sort_values(['service', 'month'])
    monthly_svc['prev_cost'] = monthly_svc.groupby('service')['unblended_cost'].shift(1)
    monthly_svc['mom_pct'] = ((monthly_svc['unblended_cost'] - monthly_svc['prev_cost'])
                              / monthly_svc['prev_cost'] * 100).round(1)

    high_growth = monthly_svc[
        (monthly_svc['mom_pct'] > threshold_pct) &
        (monthly_svc['prev_cost'].notna())
    ]

    # Exclure les services tagges growth
    growth_projects = df[df['tag_project'] == 'growth']['service'].unique()
    high_growth = high_growth[~high_growth['service'].isin(growth_projects)]

    return high_growth[['service', 'month', 'prev_cost', 'unblended_cost', 'mom_pct']]

result = detect_high_growth(df)
if len(result) > 0:
    print('Services avec croissance MoM > 50% (hors tag_project=growth) :')
    print(result.to_string(index=False))
else:
    print('Aucun service avec croissance MoM > 50% detecte')

Aucun service avec croissance MoM > 50% detecte
